# 3. Predict pricing deviations from historical contracts

The final extension tests **historical calibration transfer**, then compares it with supervised
regressors. The central scope comparison is pooled → structural-family → structural + NLP.
We do not assume that smaller groups improve forecasts. Every method sees the same held-out
positions; all outcome fitting uses training only, and validation selects settings.

Review mode reads the completed experiment. Optional recomputation stages call the same Python
functions in the order used by the CLI, writing a separate prediction-fit stage.

## 3.1 Setup, data lineage and isolated prediction outputs

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "extension/clustering/supervised/pipeline.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside the prediction-market-analysis repository.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "extension/notebooks/results/.matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(ROOT / "extension/notebooks/results/.cache"))

In [ ]:
import argparse
import json

import numpy as np
import pandas as pd
from IPython.display import display
from threadpoolctl import threadpool_limits

from extension.notebooks.support import (
    begin_stage,
    finish_stage,
    output_dir,
    prediction_sample,
    published_paths,
    read_completed,
)

published, DATA, ASSIGNMENTS, STATS, RESULTS = published_paths(ROOT)
print("Completed experiment:", RESULTS.relative_to(ROOT))
print("CSV split design:", published.get("split", "See methodology"))

In [ ]:
RUN_NAME = "completion_walkthrough"
RECOMPUTE = False  # Explicit opt-in: expensive steps write isolated notebook artifacts.
USE_REBUILT_UPSTREAM = False  # True only after completing the preceding rebuild notebooks.

model_output = output_dir(ROOT, RUN_NAME, "models")
if USE_REBUILT_UPSTREAM:
    rebuilt_measurement = output_dir(ROOT, RUN_NAME, "measurement")
    read_completed(rebuilt_measurement, stage="measurement")
    new_audit = json.loads((rebuilt_measurement / "data_audit.json").read_text())
    assignment_input = Path(new_audit["inputs"][2]["path"])
    statistics_input = rebuilt_measurement / "sufficient_statistics.csv.gz"
else:
    assignment_input, statistics_input = ASSIGNMENTS, STATS
args = argparse.Namespace(
    data=DATA, assignments=assignment_input, stats=statistics_input, output=model_output, bootstrap=2000
)

## 3.2 Build one matched population for every model
The shared loader verifies the audited cache, actual CSV split membership, contract/event
separation and frozen grouping. It excludes explicit multi-award conjunctions missed by the
original MVE-prefix filter, and requires at least 100 training contracts and 20 training events
per learned cluster. This corrected cohort is shared by all scopes, including pooled models.
The Oscar exclusion followed a misleading earlier apparent hit; it is documented rather than
presented as a new confirmatory sample. These are retrospective resolved/volume screens.

In [ ]:
from extension.clustering.supervised.historical import HistoricalCalibration, historical_predict
from extension.clustering.supervised.pipeline import load_data, select_models, train_predict

if RECOMPUTE:
    fit_manifest = begin_stage(
        ROOT,
        model_output,
        "prediction_fit",
        [args.stats, args.assignments, DATA / "master/markets.csv", DATA / "master/trades.csv"]
        + [DATA / "split" / split / "markets.csv" for split in ["train", "validation", "test"]],
        {key: str(value) if isinstance(value, Path) else value for key, value in vars(args).items()},
    )
    frame, audit = load_data(args)
    display(frame.groupby("split").agg(contracts=("ticker", "nunique"), events=("event_id", "nunique")))
else:
    display(pd.read_csv(RESULTS / "holdout_coverage.csv"))
    display(pd.read_csv(RESULTS / "combination_exclusions.csv"))
    support = pd.read_csv(RESULTS / "training_support.csv")
    display(support.groupby("eligible").agg(clusters=("cluster", "size")))

## 3.3 What does one prediction mean?
At an observed 30-cent execution, a predicted correction of +10 points implies 40% YES
probability. Retaining 10% of that correction gives 31%. The actual trade price is unchanged.
After settlement the score compares q with y; equivalently it compares predicted correction
with y-p/100. We retain all observed execution-price cells, not one fixed 72-hour forecast.
This arithmetic example is synthetic, explicitly separate from empirical results.

In [ ]:
price, raw_q, correction_fraction = 0.30, 0.40, 0.10
final_q = np.clip(price + correction_fraction * (raw_q - price), 0, 1)
display(
    pd.DataFrame(
        [
            {
                "YES_settlement": y,
                "price_probability": price,
                "raw_prediction": raw_q,
                "reduced_prediction": final_q,
                "observed_deviation_pp": 100 * (y - price),
                "predicted_deviation_pp": 100 * (final_q - price),
                "squared_error": (y - final_q) ** 2,
            }
            for y in [0, 1]
        ]
    )
)

## 3.4 Historical calibration with event-aware smoothing
Within a training group/price band, estimate the execution-weighted mean `y-p/100`.
Let W_e be the weight from training event e. Effective event support is
`E=(sum W_e)^2/sum(W_e^2)`, so repeated trades within one event cannot manufacture event support.
Use local fraction `E/(E+k)` to move from the parent residual toward the local training mean.
Pooled estimates shrink toward price (zero residual); families toward the pool; NLP clusters
toward their family. Missing group/price cells inherit the parent. This is a regularization
heuristic, not a posterior confidence statement or proof that different event IDs are independent.

The small synthetic example below tests weighting and parent fallback. It is not a fitted
model or finding from the research extract.

In [ ]:
illustration = pd.DataFrame(
    {
        "ticker": ["example-a", "example-b"],
        "event_id": ["a", "b"],
        "split": ["train", "train"],
        "family": ["example", "example"],
        "cluster": ["example/0", "example/0"],
        "p": [30, 30],
        "outcome": [0, 1],
        "n": [9, 1],
    }
)
example_model = HistoricalCalibration(width=1).fit(illustration)
display(example_model.export())
print("Smoothed probability:", 0.30 + example_model.correction(illustration, "pooled", 10, np.zeros(2)))
missing_group = illustration.assign(cluster="unseen")
np.testing.assert_array_equal(example_model.correction(missing_group, "cluster", 10, np.repeat(0.03, 2)), [0.03, 0.03])

## 3.5 Freeze the search rules before examining the new test scores
Pooled validation chooses band width 1, 5 or 10 cents and k=10,50,200,1000 or `inherit`.
Family/cluster estimates share that selected price grid and select one k per scope, sequentially
with the parent fixed. `inherit` is an explicit option to transfer no new local signal.
Training means never use validation labels; validation chooses the smoothing, and test labels
never choose either means or settings. There is no train+validation refit. The current extract
has already informed earlier development, so this is an explored retrospective holdout.

## 3.6 Train supervised comparators with the same cohort
Features: execution price and its square/cube plus title+YES-subtitle word, character, numeric,
negation and question counts. No final quote, final volume, close time or result is a predictor.
The grouping supplies the NLP component; these predictors are not supervised semantic embeddings.
Ridge uses all training cells; forest/boosting fits use at most 50,000 sampled cells per scope fit.
Training-only scaling and fixed seeds/settings match the implementation. See the methodology
for tree settings and the sampling limitation. Local fits can have substantially higher variance.

In [ ]:
if RECOMPUTE:
    with threadpool_limits(limits=2):
        held, fits = train_predict(frame, model_output)
    fits.to_csv(model_output / "model_fits.csv", index=False)
else:
    fits = pd.read_csv(RESULTS / "model_fits.csv")
display(fits.groupby(["scope", "kind"]).agg(models=("group", "size"), fitted_cells=("fitted_cells", "sum")))

## 3.7 Select corrections using validation, then freeze them
For ML, validation selects the algorithm and correction fraction per scope, including zero.
Separate partial-pooling variants shrink local predictions toward the selected pooled ML model.
For historical transfer, validation selects the hierarchy's band width/smoothing as described.
No setting is selected by which test score looks best. Display the complete candidate grids,
not just selected winners; selected validation scores are not independent confirmation.

In [ ]:
if RECOMPUTE:
    with threadpool_limits(limits=2):
        held, ml_selected = select_models(held, model_output)
        held, historical_selected = historical_predict(frame, held, model_output)
    active_results = model_output
else:
    active_results = RESULTS
    ml_selected = json.loads((RESULTS / "selected_models.json").read_text())
    historical_selected = json.loads((RESULTS / "historical_selected.json").read_text())
display(pd.DataFrame(ml_selected.values()))
display(pd.read_csv(active_results / "historical_validation_search.csv"))
display(pd.DataFrame(historical_selected.values()))

## 3.8 Inspect actual training evidence and held-out predictions
The saved training-estimates table contains raw residuals and effective support for every
candidate price grid. It contains training outcomes only. The bounded leading prediction sample
below checks the probability/correction relationship; it is not a representative performance estimate.
The full test metrics are read in notebook 04. A setting of `inherit` at all three levels means
all selected historical predictions equal price, not that true mispricing is proven zero.

In [ ]:
estimates = pd.read_csv(active_results / "historical_training_estimates.csv")
assert (estimates.effective_events <= estimates.events + 1e-8).all()
display(estimates.head(12))

if RECOMPUTE:
    held.to_parquet(model_output / "heldout_predictions.parquet", index=False)
    finish_stage(
        model_output,
        fit_manifest,
        audit=audit,
        selected=ml_selected,
        historical_selected=historical_selected,
        predictions_completed=True,
        evaluation_completed=False,
    )
    print("Prediction-fit stage complete. Notebook 04 must evaluate these predictions separately.")

prediction_columns = [
    "ticker",
    "split",
    "p",
    "outcome",
    "n",
    "market_price",
    "historical_pooled",
    "historical_family",
    "historical_cluster",
    "selected_cluster",
]
preview = prediction_sample(active_results / "heldout_predictions.parquet", prediction_columns, rows=1000)
preview["predicted_cluster_correction_pp"] = 100 * (preview.historical_cluster - preview.market_price)
preview["observed_YES_deviation_pp"] = 100 * (preview.outcome - preview.market_price)
display(preview.head(12))

## Stage interpretation
We now have frozen held-out probabilities from pooled, structural and NLP-cluster historical
baselines and supervised comparators. A completed *prediction-fit* manifest means fitting ended;
it does not mean inference/report generation is complete. Notebook 04 measures improvement,
uncertainty and direction-specific group errors before drawing conclusions.

References: [methodology](../clustering/supervised/METHODOLOGY.md),
[revision history](../clustering/supervised/CHANGES.md).